In [1]:
from transformers import pipeline
from datasets import load_dataset
import pandas as pd

In [2]:


pipe = pipeline("text-classification", model="mediabiasgroup/da-roberta-babe-ft")

ds = load_dataset("mediabiasgroup/BABE")

Device set to use cpu


README.md:   0%|          | 0.00/770 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/712k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/233k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3121 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [3]:
# check what the dataset splits look like
print(ds)
print(ds['test'].features)
print(ds['test'][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'outlet', 'label', 'topic', 'news_link', 'biased_words', 'uuid', 'type', 'label_opinion'],
        num_rows: 3121
    })
    test: Dataset({
        features: ['text', 'outlet', 'label', 'topic', 'news_link', 'biased_words', 'uuid', 'type', 'label_opinion'],
        num_rows: 1000
    })
})
{'text': Value('string'), 'outlet': Value('string'), 'label': Value('int32'), 'topic': Value('string'), 'news_link': Value('string'), 'biased_words': Value('string'), 'uuid': Value('string'), 'type': Value('string'), 'label_opinion': Value('string')}
{'text': 'As the Black Lives Matter movement grows, companies like Nike, Target, and Google have made Juneteenth a paid holiday.', 'outlet': 'Fox News', 'label': 0, 'topic': 'marriage-equality', 'news_link': 'https://www.foxnews.com/us/juneteenth-calls-increase-national-holiday?utm_source=feedburner&utm_medium=feed&utm_campaign=Feed%3A+foxnews%2Fnational+%28Internal+-+US+Latest+-+Text%29', 'b

In [5]:
sample = pipe(ds['test']['text'][:5], truncation=True)
print(sample)
print(ds['test']['label'][:5])

[{'label': 'LABEL_0', 'score': 0.8907410502433777}, {'label': 'LABEL_0', 'score': 0.7979174256324768}, {'label': 'LABEL_0', 'score': 0.9674022197723389}, {'label': 'LABEL_0', 'score': 0.9623998403549194}, {'label': 'LABEL_0', 'score': 0.8548848628997803}]
[0, 0, 0, 0, 0]


In [18]:

pd.Series(ds['train']['label']).value_counts()

1    1740
0    1381
Name: count, dtype: int64

In [7]:
texts = list(ds['test']['text'])
true_labels = ds['test']['label']

preds_raw = pipe(texts, batch_size=16, truncation=True)
preds = [0 if p['label'] == 'LABEL_0' else 1 for p in preds_raw]

print(accuracy_score(true_labels, preds))
print(f1_score(true_labels, preds))

0.823
0.816580310880829


# MBIC

In [8]:
import json

with open('10547907.json', 'r') as f:
    data = json.load(f)

# check structure
print(type(data))
print(data[0] if isinstance(data, list) else list(data.keys()))

<class 'dict'>
['access', 'created', 'custom_fields', 'deletion_status', 'files', 'id', 'is_draft', 'is_published', 'links', 'media_files', 'metadata', 'parent', 'pids', 'revision_id', 'stats', 'status', 'swh', 'updated', 'versions']


In [9]:
print(data['files'])

{'count': 3, 'enabled': True, 'entries': {'annotations.xlsx': {'access': {'hidden': False}, 'checksum': 'md5:7ee6b60316c64bc6dd7ff715bf69054b', 'ext': 'xlsx', 'id': '5962096e-7cd2-4fb8-bece-c0ba5eaf0900', 'key': 'annotations.xlsx', 'links': {'content': 'https://zenodo.org/api/records/10547907/files/annotations.xlsx/content', 'self': 'https://zenodo.org/api/records/10547907/files/annotations.xlsx'}, 'metadata': None, 'mimetype': 'application/octet-stream', 'size': 2559297, 'storage_class': 'L'}, 'annotators.csv': {'access': {'hidden': False}, 'checksum': 'md5:3f983a0fc940d4be71cf0753d66f65b6', 'ext': 'csv', 'id': '2abde708-aac1-4e88-bd78-24d01716c0f3', 'key': 'annotators.csv', 'links': {'content': 'https://zenodo.org/api/records/10547907/files/annotators.csv/content', 'self': 'https://zenodo.org/api/records/10547907/files/annotators.csv'}, 'metadata': None, 'mimetype': 'text/csv', 'size': 195055, 'storage_class': 'L'}, 'labeled_dataset.xlsx': {'access': {'hidden': False}, 'checksum': 'm

In [12]:
import requests
import pandas as pd

annotations_url = 'https://zenodo.org/api/records/10547907/files/annotations.xlsx/content'
labeled_url = 'https://zenodo.org/api/records/10547907/files/labeled_dataset.xlsx/content'

r = requests.get(annotations_url)
with open('annotations.xlsx', 'wb') as f:
    f.write(r.content)

r = requests.get(labeled_url)
with open('labeled_dataset.xlsx', 'wb') as f:
    f.write(r.content)

annotations = pd.read_excel('annotations.xlsx')
labeled = pd.read_excel('labeled_dataset.xlsx')

print(annotations.columns.tolist())
print(labeled.columns.tolist())
print(annotations.shape, labeled.shape)

['Unnamed: 0.1', 'Unnamed: 0', 'survey_record_id', 'sentence_id', 'sentence_group_id', 'created_at', 'label', 'words', 'factual', 'group_id', 'text', 'link', 'type', 'topic', 'outlet', 'age', 'gender', 'education', 'native_english_speaker', 'political_ideology', 'followed_news_outlets', 'news_check_frequency', 'survey_completed']
['Unnamed: 0', 'sentence', 'news_link', 'outlet', 'topic', 'type', 'group_id', 'num_sent', 'Label_bias', 'Label_opinion', 'article', 'biased_words4']
(17775, 23) (1700, 12)


/ihome/xli/sek188/MSTHESIS/envs/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [21]:
# one row per sentence, use consensus from labeled_dataset
sentences = labeled[['sentence', 'Label_bias']].copy()
sentences = sentences.dropna(subset=['Label_bias', 'sentence'])

# check what Label_bias looks like
print(sentences['Label_bias'].value_counts())
print(sentences['Label_bias'].dtype)

Label_bias
Biased          1018
Non-biased       533
No agreement     149
Name: count, dtype: int64
object


In [22]:
sentences['label_binary'] = sentences['Label_bias'].map({'Biased': 1, 'Non-biased': 0, 'No agreement': 2})
preds_raw = pipe(list(sentences['sentence']), batch_size=16, truncation=True)
preds = [0 if p['label'] == 'LABEL_0' else 1 for p in preds_raw]

correct = sum(p == t for p, t in zip(preds, sentences['label_binary']))
acc = correct / len(sentences)

tp = sum(p == 1 and t == 1 for p, t in zip(preds, sentences['label_binary']))
fp = sum(p == 1 and t != 1 for p, t in zip(preds, sentences['label_binary']))
fn = sum(p == 0 and t == 1 for p, t in zip(preds, sentences['label_binary']))

precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)

print(f"acc: {acc:.4f}")
print(f"f1: {f1:.4f}")

acc: 0.6800
f1: 0.7547
